# CSE476 Agentic AI & Intelligent Automation — Task 2 (CA2)
## Project Q12: Automated Grant & Scholarship Application Evaluator

**Architecture:** Multi-Agent Fan-Out / Fan-In with Microsoft Semantic Kernel, Grounded Rubric RAG, Shared State Store, and Append-Only JSONL Audit Logging.

### Agent Roster:
1. **Intake Agent:** Ingests raw dossier, validates Pydantic schemas, and initializes state.
2. **Eligibility Verification Agent:** Checks hard minimum percentage (>= 60%) and income ceiling (<= INR 600,000). Disqualifies outright or flags missing data as `NEEDS_REVIEW`.
3. **Merit Agent:** Evaluates academic percentage against a 40-point linear scaling formula.
4. **Financial Need Agent:** Maps household income into progressive need scoring brackets (35, 28, 20, 10, 0).
5. **Achievement Agent:** Evaluates co-curricular achievements, filters unverified entries, deduplicates titles/certificates, and caps at 25 points.
6. **Panel Manager Agent:** Fans in specialist results, synthesizes composite score out of 100, assigns recommendation tier, and enforces exclusion of ineligible candidates.
7. **Fairness Auditor Agent:** Audits score rationales against prohibited demographic attributes and verifies mathematical summation consistency.

In [1]:
import asyncio
import json
from decimal import Decimal
import pandas as pd

from core import ensure_sk_compatibility
from core.shared_state import StateStore
from core.audit_logger import AuditLogger
from core.scoring_config import RUBRIC_VERSION, get_configuration_hash
from rag.retriever import RubricRetriever
from orchestration.coordinator import ScholarshipCoordinator

# Apply Semantic Kernel forward-reference patch
ensure_sk_compatibility()

print(f"[INIT] System Initialized.")
print(f"[RUBRIC] Version: {RUBRIC_VERSION} | Config Hash: {get_configuration_hash()[:16]}...")

[INIT] System Initialized.
[RUBRIC] Version: 1.0.0 | Config Hash: c81e8938f1235477...


### 1. Initialize System Components and RAG Retriever

In [2]:
retriever = RubricRetriever("rag/scholarship_rubric.md")
print(f"Indexed {len(retriever.clauses)} rubric clauses successfully.")

# Test Grounded RAG Query
query = "minimum academic percentage threshold"
matches = retriever.retrieve_relevant_clauses(query, top_k=2)
for c in matches:
    print(f"-> Cited {c.clause_id}: {c.title}\n   {c.text}\n")

Indexed 11 rubric clauses successfully.
-> Cited Clause 1.1: Academic Minimum Threshold
   All candidates must possess a verified qualifying academic percentage of not less than 60.0% (sixty percent) or equivalent Grade Point Average. Candidates possessing an academic score below 60.0% are disqualified outright with status "INELIGIBLE". Academic performance cannot be waived or compensated by financial hardship or extracurricular achievements.

-> Cited Clause 2.1: Academic Performance Dimension (Weight: 40 Points)
   Academic scores are proportionally scaled from the applicant's aggregate percentage P (where 60.0 <= P <= 100.0) according to the linear formula:
Academic Score = (Percentage / 100) * 40.0
The maximum attainable academic score is 40.0 points. Scores are rounded to two decimal places.



### 2. End-to-End Evaluation of a Single Applicant Dossier

In [3]:
store = StateStore()
logger = AuditLogger("execution_audit.jsonl")
coordinator = ScholarshipCoordinator(store=store, logger=logger, retriever=retriever)

# Load sample candidate (Aarav Sharma)
with open("data/applicants.json", "r", encoding="utf-8") as f:
    applicants = json.load(f)

sample_candidate = applicants[0]
print(f"Evaluating Candidate: {sample_candidate['full_name']} (ID: {sample_candidate['applicant_id']})")

state = await coordinator.evaluate_applicant(sample_candidate)

print("\n--- EVALUATION SUMMARY ---")
print(f"Status: {state.status}")
print(f"Recommendation: {state.recommendation}")
print(f"Total Score: {state.total_score}/100.0")
print(f"  - Academic Score: {state.academic_result.score}/40.0")
print(f"  - Financial Need Score: {state.financial_result.score}/35.0 (Band: {state.financial_result.income_band})")
print(f"  - Achievement Score: {state.achievement_result.score}/25.0 (Verified: {state.achievement_result.verified_count})")
print(f"Fairness Audit Passed: {state.audit_result.passed}")

Evaluating Candidate: Aarav Sharma (ID: APP-2026-001)

--- EVALUATION SUMMARY ---
Status: ELIGIBLE_CONFIRMED
Recommendation: High Priority
Total Score: 97.00/100.0
  - Academic Score: 37.00/40.0
  - Financial Need Score: 35.0/35.0 (Band: ₹0–₹150,000)
  - Achievement Score: 25.0/25.0 (Verified: 2)
Fairness Audit Passed: True


### 3. Multi-Agent Inter-Agent Execution Trace Stream

In [4]:
print("Chronological Inter-Agent Communication Trace:")
for i, evt in enumerate(state.trace_events, 1):
    print(f"{i:02d}. [{evt.sender} -> {evt.receiver}] ({evt.event_type})")
    print(f"    {evt.payload_summary}")

Chronological Inter-Agent Communication Trace:
01. [IntakeAgent -> StateStore] (INTAKE_INITIALIZED)
    Initialized state for applicant APP-2026-001 (Aarav Sharma)
02. [EligibilityAgent -> EvaluationState] (ELIGIBILITY_EVALUATED)
    Decision: ELIGIBLE. Reason: Passed all eligibility criteria: Academic >= 60.0% and Income <= ₹600,000.00.
03. [Coordinator -> SpecialistAgents] (FAN_OUT_DISPATCH)
    Fanning out eligible applicant profile to Merit, Financial Need, and Achievement agents.
04. [MeritAgent -> EvaluationState] (ACADEMIC_EVALUATED)
    Academic Score: 37.00/40. Justification: Academic percentage 92.5% awards 37.00 out of 40.0 points (scaled linearly: (92.5/100) * 40.0).
05. [FinancialAgent -> EvaluationState] (FINANCIAL_EVALUATED)
    Financial Score: 35.0/35. Band: ₹0–₹150,000
06. [AchievementAgent -> EvaluationState] (ACHIEVEMENT_EVALUATED)
    Achievement Score: 25.0/25. Verified: 2
07. [SpecialistAgents -> PanelManagerAgent] (FAN_IN_AGGREGATION)
    Fanning in specialist s

### 4. Batch Pipeline Execution Over 17 Fictional Applicants

In [5]:
print(f"Starting concurrent batch evaluation of {len(applicants)} candidate dossiers...")
all_states = await coordinator.evaluate_batch(applicants)
print(f"Completed batch evaluation of {len(all_states)} applicants.")

leaderboard = await coordinator.get_ranked_leaderboard()

Starting concurrent batch evaluation of 17 candidate dossiers...
Completed batch evaluation of 17 applicants.


### 5. Official Ranked Committee Leaderboard

In [6]:
confirmed_df = pd.DataFrame(leaderboard["confirmed_awardees"])[
    ["applicant_id", "full_name", "academic_score", "financial_score", "achievement_score", "total_score", "recommendation"]
]
confirmed_df.columns = ["ID", "Candidate Name", "Acad (40)", "Need (35)", "Ach (25)", "Total (100)", "Recommendation"]
print("=== CONFIRMED SCHOLARSHIP AWARDEES (Ranked Descending) ===")
confirmed_df

=== CONFIRMED SCHOLARSHIP AWARDEES (Ranked Descending) ===
          ID Candidate Name  Acad (40)  Need (35)  Ach (25)  Total (100) Recommendation
APP-2026-016    Pooja Hegde       38.0       35.0      25.0         98.0  High Priority
APP-2026-001   Aarav Sharma       37.0       35.0      25.0         97.0  High Priority
APP-2026-002   Bhavya Patel       32.8       35.0       9.0         76.8    Recommended
APP-2026-003   Chirag Verma       31.4       28.0      10.0         69.4    Recommended
APP-2026-017     Rohit Nair       31.6       28.0       6.0         65.6    Recommended


### 6. Disqualified and Needs Review Pools (Exclusion Enforced)

In [7]:
ineligible_df = pd.DataFrame(leaderboard["ineligible_disqualified"])[
    ["applicant_id", "full_name", "academic_percentage", "household_income", "disqualification_reasons", "status"]
]
print("=== INELIGIBLE / DISQUALIFIED CANDIDATES (Zero Award Slots) ===")
ineligible_df

=== INELIGIBLE / DISQUALIFIED CANDIDATES (Zero Award Slots) ===
applicant_id       full_name  academic_percentage  household_income                                                                                                         disqualification_reasons     status
APP-2026-007    Gaurav Joshi                 54.0          110000.0                                                                [Academic percentage (54.0%) is below minimum threshold of 60.0%] INELIGIBLE
APP-2026-008 Harini Krishnan                 89.0          850000.0                                                                  [Household income (₹850,000.00) exceeds ceiling of ₹600,000.00] INELIGIBLE
APP-2026-009       Ishan Sen                 52.0          950000.0 [Academic percentage (52.0%) is below minimum threshold of 60.0%, Household income (₹950,000.00) exceeds ceiling of ₹600,000.00] INELIGIBLE


### 7. Append-Only Audit Log Verification (`execution_audit.jsonl`)

In [8]:
logs = logger.read_all_logs()
print(f"Total Append-Only Audit Records: {len(logs)}")
sample_log = logs[-1]
print("Latest Log Entry Sample:")
print(json.dumps(sample_log, indent=2))

Total Append-Only Audit Records: 425
Latest Log Entry Sample:
{
  "timestamp": "2026-11-14T16:58:50.902108+00:00",
  "applicant_id": "APP-2026-017",
  "agent_name": "Coordinator",
  "tool_name": "pipeline_complete",
  "input_summary": "End-to-end evaluation for APP-2026-017",
  "output_summary": "Final Status: ELIGIBLE_CONFIRMED, Recommendation: Recommended, Score: 65.60",
  "execution_status": "SUCCESS",
  "duration_seconds": 0.2349,
  "error_details": null,
  "rubric_version": "1.0.0",
  "config_hash": "c81e8938f123547732f8f0dd4cd6222475761b4db6633f6d002eec6e25416fea",
  "metadata": {}
}
